# NumPy Orta Düzey

Bu notebook, "NumPy Temelleri" notebook'unun devamıdır ve daha **orta düzey** konuları satır satır Türkçe açıklamalarla ele alır.

İçerik:
1. View (Görünüm) ve Copy (Kopya) Farkı
2. Fancy Indexing (Gelişmiş İndeksleme)
3. np.where ile Koşullu İşlemler
4. Sıralama: sort, argsort, unique
5. Array Birleştirme ve Bölme (concatenate, stack, split)
6. Eksen (axis) Kavramını Derinlemesine Anlamak
7. Vektörleştirme ile Performans Karşılaştırması
8. Universal Functions (ufunc) ve Kendi ufunc'ını Yazmak
9. Yapılandırılmış (Structured) Array'ler
10. Doğrusal Cebir: Özdeğer, Özvektör ve Doğrusal Denklem Çözme
11. Dosyaya Kaydetme ve Dosyadan Okuma
12. Maskeleme ile Veri Temizleme (Gerçekçi Mini Örnek)


In [1]:
import numpy as np

# Bu notebook boyunca sonuçların tekrarlanabilir olması için seed sabitliyoruz
np.random.seed(0)

## 1. View (Görünüm) ve Copy (Kopya) Farkı

NumPy'da dilimleme (slicing) işlemi **view** (görünüm) döndürür; yani orijinal veriyle aynı belleği paylaşır.
Bunu bilmemek, istemeden orijinal veriyi değiştirmenize yol açabilir.

In [2]:
orijinal = np.array([1, 2, 3, 4, 5])   # 1D örnek array

dilim = orijinal[1:4]     # slicing yaptık; bu bir VIEW'dur, yeni bellek ayrılmaz
dilim[0] = 99             # dilim üzerinde değişiklik yapıyoruz

print("Dilim   :", dilim)      # [99  3  4]
print("Orijinal:", orijinal)   # [ 1 99  3  4  5]  -> orijinal de değişti! çünkü aynı belleği paylaşıyorlar

Dilim   : [99  3  4]
Orijinal: [ 1 99  3  4  5]


In [3]:
orijinal = np.array([1, 2, 3, 4, 5])

kopya = orijinal[1:4].copy()   # .copy() ile bağımsız bir KOPYA oluşturuyoruz; artık ayrı bellek kullanılır
kopya[0] = 99                  # kopya üzerinde değişiklik yapıyoruz

print("Kopya   :", kopya)      # [99  3  4]
print("Orijinal:", orijinal)   # [1 2 3 4 5]  -> orijinal ETKİLENMEDİ, çünkü kopya bağımsız bellekte

Kopya   : [99  3  4]
Orijinal: [1 2 3 4 5]


In [4]:
# base attribute'u ile bir array'in view mi yoksa kendi belleğinde mi olduğunu kontrol edebiliriz
a = np.arange(10)
b = a[2:5]          # view
c = a[2:5].copy()   # copy

print("b.base is a :", b.base is a)   # True -> b, a'nın belleğini paylaşıyor (view)
print("c.base is a :", c.base is a)   # False -> c bağımsız bir kopya

b.base is a : True
c.base is a : False


## 2. Fancy Indexing (Gelişmiş İndeksleme)

Normal slicing'den farklı olarak, bir array'e **liste veya array** vererek istediğimiz indekslerdeki elemanları
istediğimiz sırayla seçebiliriz. Bu işlem her zaman bir **copy** döndürür.

In [5]:
a = np.array([10, 20, 30, 40, 50])

indeksler = [0, 2, 4]          # seçmek istediğimiz indeksleri bir liste olarak tanımlıyoruz
print(a[indeksler])            # bu indekslerdeki elemanları seçer -> [10 30 50]

# İndeksleri istediğimiz sırada da verebiliriz, hatta tekrar edebiliriz
print(a[[4, 4, 0]])            # -> [50 50 10]

[10 30 50]
[50 50 10]


In [6]:
# 2 boyutlu array'lerde fancy indexing ile hem satır hem sütun seçimi yapılabilir
m = np.array([[1,  2,  3],
              [4,  5,  6],
              [7,  8,  9],
              [10, 11, 12]])

satirlar = [0, 2, 3]           # almak istediğimiz satır indeksleri
print(m[satirlar])             # bu satırların tamamını seçer

# Satır ve sütun indekslerini eş zamanlı vererek belirli noktaları seçebiliriz
satir_idx = [0, 1, 2]
sutun_idx = [2, 0, 1]
print(m[satir_idx, sutun_idx]) # (0,2), (1,0), (2,1) noktalarındaki elemanları seçer -> [3 4 8]

[[ 1  2  3]
 [ 7  8  9]
 [10 11 12]]
[3 4 8]


## 3. np.where ile Koşullu İşlemler

`np.where(koşul, doğruysa, yanlışsa)` fonksiyonu, koşula göre iki farklı değer arasında seçim yapmamızı sağlar.
Bu, döngü yazmadan koşullu dönüşüm yapmanın en hızlı yoludur.

In [7]:
notlar = np.array([45, 60, 78, 32, 90, 55])

# 50'nin altındaki notları "Kaldı", üstündekileri "Geçti" olarak etiketliyoruz
sonuc = np.where(notlar >= 50, "Geçti", "Kaldı")
print(sonuc)

# Sadece koşul veren kullanım: koşula uyan elemanların İNDEKSLERİNİ döndürür
gecen_indeksler = np.where(notlar >= 50)
print("Geçenlerin indeksleri:", gecen_indeksler[0])   # [1 2 4 5]

['Kaldı' 'Geçti' 'Geçti' 'Kaldı' 'Geçti' 'Geçti']
Geçenlerin indeksleri: [1 2 4 5]


In [8]:
# np.where negatif sayıları 0 ile değiştirmek gibi pratik durumlarda çok kullanılır (ReLU benzeri)
veriler = np.array([-3, 5, -1, 8, -7, 2])
temiz_veriler = np.where(veriler < 0, 0, veriler)   # negatifse 0 yap, değilse olduğu gibi bırak
print(temiz_veriler)   # [0 5 0 8 0 2]

[0 5 0 8 0 2]


## 4. Sıralama: sort, argsort, unique

NumPy, array'leri sıralamak ve tekrarsız (unique) değerleri bulmak için hazır fonksiyonlar sunar.

In [9]:
a = np.array([5, 2, 8, 1, 9, 3])

print("sort    :", np.sort(a))       # elemanları küçükten büyüğe sıralar (yeni array döndürür) -> [1 2 3 5 8 9]
print("argsort :", np.argsort(a))    # sıralamayı yapacak İNDEKS sırasını döndürür -> [3 1 5 0 2 4]

# argsort'un mantığı: a[argsort(a)] her zaman sıralanmış array'i verir
print("Doğrulama:", a[np.argsort(a)])

sort    : [1 2 3 5 8 9]
argsort : [3 1 5 0 2 4]
Doğrulama: [1 2 3 5 8 9]


In [10]:
# argsort'u pratikte, bir diziye göre başka bir diziyi sıralamak için kullanırız
isimler = np.array(["Ayşe", "Mehmet", "Can", "Zeynep"])
yaslar  = np.array([25, 32, 19, 41])

sira = np.argsort(yaslar)          # yaşları küçükten büyüğe sıralayacak indeks sırasını bulur
print("Yaşa göre sıralı isimler:", isimler[sira])   # isimleri aynı sıraya göre yeniden düzenler
print("Sıralı yaşlar          :", yaslar[sira])

Yaşa göre sıralı isimler: ['Can' 'Ayşe' 'Mehmet' 'Zeynep']
Sıralı yaşlar          : [19 25 32 41]


In [11]:
a = np.array([1, 2, 2, 3, 3, 3, 4])

tekil_degerler = np.unique(a)      # tekrar eden elemanları kaldırıp tekil değerleri döndürür
print("unique         :", tekil_degerler)

# return_counts=True ile her tekil değerin kaç kez geçtiğini de öğrenebiliriz
degerler, sayilar = np.unique(a, return_counts=True)
print("Değerler:", degerler)
print("Sayılar :", sayilar)   # her değerin dizide kaç kez tekrarlandığı

unique         : [1 2 3 4]
Değerler: [1 2 3 4]
Sayılar : [1 2 3 1]


## 5. Array Birleştirme ve Bölme

Birden fazla array'i birleştirmek veya bir array'i parçalara ayırmak için kullanılan fonksiyonlardır.

In [12]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("concatenate:", np.concatenate([a, b]))   # array'leri uç uca ekler -> [1 2 3 4 5 6]
print("vstack     :\n", np.vstack([a, b]))       # array'leri alt alta (dikey) yığar -> 2x3 matris
print("hstack     :", np.hstack([a, b]))         # array'leri yan yana (yatay) birleştirir -> [1 2 3 4 5 6]

concatenate: [1 2 3 4 5 6]
vstack     :
 [[1 2 3]
 [4 5 6]]
hstack     : [1 2 3 4 5 6]


In [13]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
v = np.array([[7, 8, 9]])   # matrise eklemek için 2D olarak tanımladık

print("Satır ekleme:\n", np.vstack([m, v]))       # yeni bir satır olarak alta ekler

sutun = np.array([[10], [11]])   # 2 satırlı bir sütun vektörü
print("Sütun ekleme:\n", np.hstack([m, sutun]))   # yeni bir sütun olarak sağa ekler

Satır ekleme:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Sütun ekleme:
 [[ 1  2  3 10]
 [ 4  5  6 11]]


In [14]:
a = np.arange(9)   # 0'dan 8'e kadar 9 elemanlı array

parcalar = np.split(a, 3)   # array'i 3 EŞİT parçaya böler
for i, parca in enumerate(parcalar):
    print(f"Parça {i}:", parca)

# array_split, eşit bölünemeyen durumlarda da çalışır (parçaları mümkün olduğunca eşit dağıtır)
esit_olmayan_parcalar = np.array_split(np.arange(10), 3)
print("Eşit olmayan bölme:", esit_olmayan_parcalar)

Parça 0: [0 1 2]
Parça 1: [3 4 5]
Parça 2: [6 7 8]
Eşit olmayan bölme: [array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]


## 6. Eksen (axis) Kavramını Derinlemesine Anlamak

`axis` parametresi başlangıçta kafa karıştırıcı olabilir. Kural şudur:
**axis=0 dikey yönde (satırlar boyunca aşağı) hareket eder, axis=1 yatay yönde (sütunlar boyunca sağa) hareket eder.**
Yani `axis=0` dediğimizde, o eksen "çökertilir" (elenir) ve geriye kalan boyut üzerinden işlem yapılır.

In [15]:
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# axis=0: satırlar boyunca ilerler -> her SÜTUN için tek bir sonuç üretir (3 sütun -> 3 sonuç)
print("axis=0 toplam (sütun bazlı):", m.sum(axis=0))   # [12 15 18]

# axis=1: sütunlar boyunca ilerler -> her SATIR için tek bir sonuç üretir (3 satır -> 3 sonuç)
print("axis=1 toplam (satır bazlı):", m.sum(axis=1))   # [ 6 15 24]

# axis belirtilmezse tüm elemanlar tek bir skalere indirgenir
print("axis yok (genel toplam)   :", m.sum())          # 45

axis=0 toplam (sütun bazlı): [12 15 18]
axis=1 toplam (satır bazlı): [ 6 15 24]
axis yok (genel toplam)   : 45


In [16]:
# 3 boyutlu bir örnekle axis mantığını pekiştirelim
kup = np.arange(24).reshape(2, 3, 4)   # (2, 3, 4) boyutunda 3D array: 2 katman, 3 satır, 4 sütun
print("Şekil:", kup.shape)

print("axis=0 ile toplam şekli:", kup.sum(axis=0).shape)  # (3, 4) -> katman ekseni yok oldu
print("axis=1 ile toplam şekli:", kup.sum(axis=1).shape)  # (2, 4) -> satır ekseni yok oldu
print("axis=2 ile toplam şekli:", kup.sum(axis=2).shape)  # (2, 3) -> sütun ekseni yok oldu

Şekil: (2, 3, 4)
axis=0 ile toplam şekli: (3, 4)
axis=1 ile toplam şekli: (2, 4)
axis=2 ile toplam şekli: (2, 3)


## 7. Vektörleştirme ile Performans Karşılaştırması

NumPy'ın asıl gücü, döngü yerine **vektörleştirilmiş** (eleman bazlı, C dilinde çalışan) işlemler kullanmaktan gelir.
Aşağıda aynı işlemi saf Python döngüsüyle ve NumPy vektörleştirmesiyle karşılaştırıyoruz.

In [17]:
import time

n = 1_000_000
liste = list(range(n))          # 1 milyon elemanlı Python listesi
array = np.arange(n)            # 1 milyon elemanlı NumPy array'i

# --- Saf Python döngüsü ile kareleri alma ---
baslangic = time.time()                       # zaman ölçümünü başlatıyoruz
liste_kareler = [x ** 2 for x in liste]       # her elemanı döngüyle tek tek karesini alıyoruz
python_suresi = time.time() - baslangic       # geçen süreyi hesaplıyoruz

# --- NumPy vektörleştirmesi ile kareleri alma ---
baslangic = time.time()
array_kareler = array ** 2                    # tüm elemanların karesini TEK SATIRDA, döngüsüz alıyoruz
numpy_suresi = time.time() - baslangic

print(f"Python döngüsü süresi : {python_suresi:.4f} saniye")
print(f"NumPy vektörleştirme  : {numpy_suresi:.4f} saniye")
print(f"NumPy yaklaşık {python_suresi / numpy_suresi:.1f} kat daha hızlı")

Python döngüsü süresi : 0.2388 saniye
NumPy vektörleştirme  : 0.0425 saniye
NumPy yaklaşık 5.6 kat daha hızlı


## 8. Universal Functions (ufunc)

`ufunc`, array'lerin her elemanına ayrı ayrı, hızlı ve vektörleştirilmiş şekilde uygulanan fonksiyonlardır.
NumPy'ın `sin`, `exp`, `sqrt` gibi birçok hazır ufunc'ı vardır; kendi fonksiyonumuzu da ufunc'a çevirebiliriz.

In [18]:
a = np.array([0, np.pi/2, np.pi])   # trigonometrik örnek için radyan değerleri

print("sin  :", np.sin(a))     # her elemanın sinüsünü alır
print("sqrt :", np.sqrt([1, 4, 9, 16]))  # her elemanın karekökünü alır -> [1. 2. 3. 4.]
print("exp  :", np.exp([0, 1, 2]))       # her eleman için e^x hesaplar

sin  : [0.0000000e+00 1.0000000e+00 1.2246468e-16]
sqrt : [1. 2. 3. 4.]
exp  : [1.         2.71828183 7.3890561 ]


In [19]:
# Kendi yazdığımız normal bir Python fonksiyonunu np.vectorize ile ufunc benzeri hale getirebiliriz
def kendi_fonksiyonum(x):
    # eğer sayı çift ise ikiye böl, tek ise 3 katını al ve 1 ekle (Collatz kuralı gibi)
    if x % 2 == 0:
        return x / 2
    else:
        return x * 3 + 1

vektorize_fonksiyon = np.vectorize(kendi_fonksiyonum)  # fonksiyonu array'e uygulanabilir hale getiriyoruz

a = np.array([1, 2, 3, 4, 5, 6])
print(vektorize_fonksiyon(a))   # her elemana fonksiyonu tek tek ama vektörleştirilmiş sözdizimiyle uygular

# Not: np.vectorize performans için değil, KOD OKUNABİLİRLİĞİ için kullanılır;
# arka planda yine Python döngüsü çalışır.

[ 4  1 10  2 16  3]


## 9. Yapılandırılmış (Structured) Array'ler

Bazen her satırın farklı tipte sütunlara (isim, yaş, boy gibi) sahip olmasını isteriz.
NumPy'da bunun için **structured array** kullanılır; bu, basit bir veri tablosuna benzer.

In [20]:
# Her elemanın "isim" (string), "yas" (int) ve "boy" (float) alanlarına sahip olacağını tanımlıyoruz
veri_tipi = np.dtype([("isim", "U10"), ("yas", "i4"), ("boy", "f4")])
# U10 -> en fazla 10 karakterlik unicode string, i4 -> 4 byte'lık integer, f4 -> 4 byte'lık float

kisiler = np.array([
    ("Ahmet", 28, 1.78),
    ("Elif", 34, 1.65),
    ("Burak", 22, 1.82)
], dtype=veri_tipi)   # tanımladığımız yapıya göre array'i oluşturuyoruz

print(kisiler)
print("İsimler       :", kisiler["isim"])   # sadece "isim" sütununu seçer
print("Ortalama yaş  :", kisiler["yas"].mean())    # yaş sütununun ortalamasını alır
print("En uzun boylu :", kisiler[kisiler["boy"].argmax()])  # en yüksek boy değerine sahip kişiyi bulur

[('Ahmet', 28, 1.78) ('Elif', 34, 1.65) ('Burak', 22, 1.82)]
İsimler       : ['Ahmet' 'Elif' 'Burak']
Ortalama yaş  : 28.0
En uzun boylu : ('Burak', 22, 1.82)


## 10. Doğrusal Cebir: Özdeğer, Özvektör ve Doğrusal Denklem Çözme

`np.linalg` modülü, temel işlemlerin ötesinde özdeğer/özvektör hesaplama ve doğrusal denklem sistemlerini
çözme gibi daha ileri seviye işlevler de sunar.

In [21]:
A = np.array([[4, 2],
              [1, 3]])   # örnek kare matris

# Özdeğer (eigenvalue) ve özvektörleri (eigenvector) hesaplıyoruz
ozdegerler, ozvektorler = np.linalg.eig(A)

print("Özdeğerler :", ozdegerler)
print("Özvektörler:\n", ozvektorler)

Özdeğerler : [5. 2.]
Özvektörler:
 [[ 0.89442719 -0.70710678]
 [ 0.4472136   0.70710678]]


In [22]:
# Ax = b şeklindeki bir doğrusal denklem sistemini çözüyoruz
# Örnek: 3x + y = 9  ve  x + 2y = 8
A = np.array([[3, 1],
              [1, 2]])   # katsayılar matrisi
b = np.array([9, 8])     # sağ taraftaki sabitler

cozum = np.linalg.solve(A, b)   # denklem sistemini çözer (matris tersini almaktan daha kararlı ve hızlıdır)
print("x =", cozum[0], ", y =", cozum[1])

# Sonucu doğrulayalım: A @ cozum, b'ye eşit olmalı
print("Doğrulama (A @ x):", A @ cozum)

x = 2.0 , y = 3.0
Doğrulama (A @ x): [9. 8.]


## 11. Dosyaya Kaydetme ve Dosyadan Okuma

Büyük array'leri her seferinde yeniden hesaplamak yerine, NumPy'ın kendi ikili (binary) formatında
diske kaydedip tekrar hızlıca okuyabiliriz.

In [23]:
a = np.arange(20).reshape(4, 5)   # örnek bir array oluşturduk

np.save("ornek_array.npy", a)     # array'i .npy uzantılı ikili formatta diske kaydeder
yuklenen = np.load("ornek_array.npy")  # kaydedilen dosyayı geri yükler

print("Kaydedilen ile yüklenen aynı mı?:", np.array_equal(a, yuklenen))  # True olmalı

# Birden fazla array'i tek dosyada saklamak istersek np.savez kullanırız
np.savez("coklu_array.npz", birinci=a, ikinci=a * 2)
veri = np.load("coklu_array.npz")
print("Dosyadaki anahtarlar:", veri.files)   # ['birinci', 'ikinci']
print("İkinci array:\n", veri["ikinci"])

Kaydedilen ile yüklenen aynı mı?: True
Dosyadaki anahtarlar: ['birinci', 'ikinci']
İkinci array:
 [[ 0  2  4  6  8]
 [10 12 14 16 18]
 [20 22 24 26 28]
 [30 32 34 36 38]]


## 12. Maskeleme ile Veri Temizleme (Gerçekçi Mini Örnek)

Son olarak, öğrendiklerimizi birleştirerek eksik ve aykırı (outlier) değerler içeren basit bir veri setini
NumPy ile temizleyelim.

In [24]:
# Elimizde sıcaklık ölçümleri var; np.nan eksik veriyi, -999 ise hatalı sensör okumasını temsil ediyor
sicakliklar = np.array([22.5, 23.1, np.nan, 21.8, -999, 24.0, 22.9, np.nan, 150.0, 23.5])

print("Ham veri:", sicakliklar)

# 1) Eksik (nan) değerleri tespit ediyoruz
eksik_maske = np.isnan(sicakliklar)
print("Eksik değer sayısı:", eksik_maske.sum())

# 2) Mantıksal olmayan (aykırı) değerleri belirliyoruz: örneğin -50 ile 50 derece dışındakiler hatalıdır
gecerli_maske = (sicakliklar > -50) & (sicakliklar < 50) & (~np.isnan(sicakliklar))
# ~ işareti "değil" (NOT) anlamına gelir; yani nan OLMAYANLARI seçiyoruz

temiz_veri = sicakliklar[gecerli_maske]   # sadece geçerli olan (nan ve aykırı olmayan) değerleri filtreliyoruz
print("Temiz veri     :", temiz_veri)
print("Temiz ortalama :", temiz_veri.mean())
print("Kaç ölçüm elendi:", sicakliklar.size - temiz_veri.size)

Ham veri: [  22.5   23.1    nan   21.8 -999.    24.    22.9    nan  150.    23.5]
Eksik değer sayısı: 2
Temiz veri     : [22.5 23.1 21.8 24.  22.9 23.5]
Temiz ortalama : 22.96666666666667
Kaç ölçüm elendi: 4


## Özet

Bu orta düzey notebook'ta öğrendiklerimiz:

- **View vs Copy**: slicing bellek paylaşır, `.copy()` bağımsız kopya oluşturur
- **Fancy indexing**: liste/array ile istenen sıradaki elemanları seçme
- `np.where` ile koşullu dönüşüm
- `sort`, `argsort`, `unique` ile sıralama ve tekilleştirme
- `concatenate`, `vstack`, `hstack`, `split` ile birleştirme/bölme
- `axis` parametresinin 2D ve 3D array'lerde nasıl çalıştığı
- Vektörleştirmenin saf Python döngülerine göre performans avantajı
- `ufunc` kavramı ve `np.vectorize`
- Structured array'ler ile tablo benzeri veri tutma
- `np.linalg.eig` ve `np.linalg.solve` ile ileri doğrusal cebir
- `np.save` / `np.load` ile array'leri diske kaydetme
- Gerçekçi bir veri temizleme senaryosunda maskeleme

Sonraki adım için önerilen ileri düzey konular: `np.einsum`, bellek düzeni (C-order/F-order),
`strides`, `memmap` ile büyük dosyalarla çalışma, ve NumPy ile pandas/scipy entegrasyonu.
